# 01. OpenAI Agents SDK — IT Help Desk Agent (Local Function Tools)

This script demonstrates the **OpenAI Agents SDK** (`pip install openai-agents`, imported as `agents`) — an
open-source, client-side agent orchestration framework from OpenAI. It builds a small "IT Help Desk" agent
that answers questions by calling **local Python functions** decorated with `@function_tool`.

This is deliberately the *first* file in the section, and it is **not** an Azure AI Foundry example — no
`azure-ai-projects`, no Foundry project endpoint. It exists to give you a baseline mental model of "agent +
tools + a run loop" using a lightweight, fully local SDK, before the rest of the section moves to Azure AI
Foundry's **Agent Service** (`02_prompt_agent.py` onward), which hosts the very same *concept* (an LLM + a
system prompt + tools) as a managed, versioned, server-side resource instead of a few lines of local Python.

Compare this file's `Agent(...)` + `Runner.run(...)` pattern side-by-side with `02_prompt_agent.py`'s
`PromptAgentDefinition` + `client.agents.create_version(...)` pattern — same idea, two very different
deployment models.

**Difficulty:** Beginner


## Prerequisites

**Pip packages** (check imports: `asyncio` is stdlib; `agents` is the OpenAI Agents SDK):
- `openai-agents` — **not currently in the repo's root `requirements.txt`**, install with:
  ```bash
  pip install openai-agents
  ```
- Uses the `openai` package under the hood (already in root `requirements.txt`).

**Azure resource(s) required:** None — this script talks directly to the OpenAI API, not Azure AI Foundry.

**Environment variables** (add to the root `.env`):
- `OPENAI_API_KEY` — already present in this repo's root `.env` per `CLAUDE.md`. The Agents SDK reads this
  automatically from the environment; you never pass it explicitly in code.

No Azure credential, endpoint, or `az login` is needed for this particular file.


## What You'll Learn

- How the OpenAI Agents SDK models an "agent" as a Python object (`Agent(name=..., model=..., instructions=..., tools=...)`)
- How `@function_tool` turns an ordinary Python function into a callable tool the model can invoke, using its docstring and type hints as the tool's description/schema
- How `Runner.run(...)` drives the full agentic loop (send prompt → model decides to call a tool → tool executes locally → model produces a final answer) in one `await`
- Why this **local, code-first** agent pattern is a useful contrast to Azure AI Foundry's **hosted, versioned** Agent Service used in the rest of this section


### Setup

Standard imports, plus a `dotenv` load so the SDK's implicit `OPENAI_API_KEY` lookup is fed from the repo's
root `.env` rather than requiring you to export it in your shell every session. The original script has no
hardcoded secrets to refactor — the Agents SDK never takes a key as a literal in code, it just reads the
environment — so `load_dotenv()` is the only environment-driven addition needed here.

- **💡 Exam tip:** AI-102 does not cover the OpenAI Agents SDK directly (it's an OpenAI-ecosystem tool, not
  an Azure service), but understanding "agent = model + instructions + tools + a run loop" as a concept
  transfers directly to how Azure AI Foundry's `PromptAgentDefinition` is structured later in this section.
- **🔄 Alternatives:** Azure AI Foundry's Agent Service (`azure-ai-projects` + `azure-ai-agents`, seen from
  `02_prompt_agent.py` on) gives you the same agent-plus-tools model as a persisted, versioned, server-hosted
  resource with built-in tools like `WebSearchTool` and `FileSearchTool` — no local run-loop code required.


In [ ]:
import asyncio
import os

from dotenv import load_dotenv
from agents import Agent, function_tool, Runner

# Loads OPENAI_API_KEY (and anything else in the root .env) into the process environment.
# The Agents SDK picks up OPENAI_API_KEY implicitly — no explicit client construction needed.
load_dotenv()


### Defining the tools

Each `@function_tool`-decorated function becomes something the model can choose to call. The Agents SDK
inspects the function's **type hints** (for the JSON schema of its parameters) and its **docstring** (for the
tool description shown to the model) automatically — you don't hand-write a JSON schema like you will later
with Azure AI Foundry's `FunctionTool` (see `05_IT_HelpDesk_agent.py`).

- `get_password_reset_steps()` and `get_vpn_troubleshooting_steps()` take no arguments and return canned
  instructional text.
- `get_software_install_guide(software_name: str)` takes one string argument and looks it up in a small
  in-memory dictionary, falling back to a "no guide found" message for anything not in the dict.

- **💡 Exam tip:** This "function calling" pattern — model requests a function by name with JSON arguments,
  caller executes it, caller returns the result to the model — is the exact same contract Azure AI Foundry's
  `FunctionTool` uses (see `05_IT_HelpDesk_agent.py` / `07_helpdesk_client.py`). The Agents SDK just automates
  the "caller executes it" step for you; Azure AI Foundry's `FunctionTool` leaves that step to your own code.
- **🔄 Alternatives:** Instead of local Python functions, tools can be built-in/hosted (web search, code
  interpreter, file search) that execute *server-side* with no local code at all — see `04_agent_web_search.py`.


In [ ]:
@function_tool
def get_password_reset_steps() -> str:
    """Returns the steps to reset a company account password."""
    return (
        "To reset your password: "
        "1. Go to https://accounts.company.com/reset. "
        "2. Enter your company email address. "
        "3. Check your email for a reset link (expires in 15 minutes). "
        "4. Follow the link and choose a new password. "
        "5. If the link does not arrive, check your spam folder or contact IT."
    )


@function_tool
def get_vpn_troubleshooting_steps() -> str:
    """Returns troubleshooting steps for VPN connection issues."""
    return (
        "VPN troubleshooting steps: "
        "1. Confirm you are connected to the internet before launching the VPN client. "
        "2. Check that your VPN client is up to date — version 4.2 or later is required. "
        "3. Try disconnecting and reconnecting. "
        "4. If the issue persists, restart your machine and try again. "
        "5. If you are on a public network, some ports may be blocked — try a mobile hotspot. "
        "6. Contact IT if the problem continues after these steps."
    )


@function_tool
def get_software_install_guide(software_name: str) -> str:
    """Returns installation instructions for a specified software package."""
    guides = {
        "slack": "To install Slack: visit https://slack.com/downloads, download the installer for your OS, and sign in with your company email.",
        "zoom": "To install Zoom: visit https://zoom.us/download, download Zoom Desktop Client, and sign in via SSO using your company domain.",
        "vscode": "To install VS Code: visit https://code.visualstudio.com, download the installer, and follow the setup wizard.",
    }
    name = software_name.lower()
    return guides.get(
        name,
        f"No installation guide found for '{software_name}'. Please contact IT for assistance."
    )


### Constructing the agent

`Agent(...)` bundles a name, a model ID, a system prompt (`instructions`), and the list of tools it's allowed
to call. Note the model name `"gpt-5.4-mini"` is whatever this course's OpenAI account had available at
recording time — swap it for a model your own `OPENAI_API_KEY` actually has access to (e.g. `"gpt-4o-mini"`)
if you run this for real. We pull it from an env var with that value as the default so you can override it
without editing code.

- **💡 Exam tip:** The `instructions` field here is functionally identical to the `instructions` parameter of
  Azure AI Foundry's `PromptAgentDefinition` — both are the agent's system prompt. Knowing that "system
  prompt" and "instructions" are the same concept under different SDK names helps when reading exam questions
  that mix vocabulary from different Azure AI product surfaces.
- **🔄 Alternatives:** You could omit `tools` entirely for a pure chat agent, or pass a much larger tool list —
  Azure AI Foundry's `ToolSet` (covered in `11_azure_ai_foundry/03_toolbox`) is the equivalent pattern for
  bundling many tools together in the Foundry SDK.


In [ ]:
MODEL_NAME = os.getenv("OPENAI_AGENTS_MODEL", "gpt-5.4-mini")

helpdesk_agent = Agent(
    name="IT Help Desk Agent",
    model=MODEL_NAME,
    instructions=(
        "You are a helpful IT support assistant for a company. "
        "When a user asks a question, use the available tools to find the answer. "
        "Always use a tool before responding — do not answer from memory alone. "
        "Keep your responses clear and concise."
    ),
    tools=[
        get_password_reset_steps,
        get_vpn_troubleshooting_steps,
        get_software_install_guide,
    ],
)


### Running the agent

`Runner.run(agent, input=query)` is an `async` call that drives the full loop: send the query and available
tools to the model, let the model decide whether/which tool to call, execute the chosen tool locally, feed
the result back to the model, and return the final text in `result.final_output`. We loop over three sample
questions, one per tool, and print the agent's answer for each.

- **💡 Exam tip:** This request/tool-call/tool-result/final-answer cycle is exactly the "agentic loop" concept
  AI-102 expects you to recognize regardless of SDK — you'll see it again explicitly (with manual code) in
  `07_helpdesk_client.py` when using Azure AI Foundry's `FunctionTool`, where *you* write the loop instead of
  `Runner.run` writing it for you.
- **🔄 Alternatives:** `Runner.run` is async; the SDK also offers a synchronous `Runner.run_sync` for simpler
  scripts. There's also a streaming variant for token-by-token output, mirroring the streaming patterns you'll
  see elsewhere in this repo (`08_ai_apps/15_streaming`).


In [ ]:
async def main():
    queries = [
        "How do I reset my password?",
        "My VPN keeps disconnecting. What should I do?",
        "Can you tell me how to install Slack?",
    ]

    for query in queries:
        print(f"\nUser: {query}")
        result = await Runner.run(helpdesk_agent, input=query)
        print(f"Agent: {result.final_output}")


if __name__ == "__main__":
    asyncio.run(main())


## Summary

This notebook builds a fully local IT help-desk agent using the OpenAI Agents SDK: three `@function_tool`
functions, an `Agent` wired up with a system prompt and those tools, and an async loop that runs three sample
questions through `Runner.run`. For each question the model should pick the matching tool, the SDK executes
it in-process, and the final printed line is the model's natural-language answer built from the tool's
output — a working demonstration of "an LLM with tools" with zero Azure infrastructure involved.

Keep this pattern in mind as a baseline: everything from `02_prompt_agent.py` onward reimplements the same
"model + instructions + tools" idea as a managed Azure AI Foundry resource instead of local Python objects.


## Try It Yourself

1. **Easy:** Add a fourth tool, e.g. `get_printer_setup_steps()`, and add a matching query to the `queries`
   list to see the agent pick it up.
2. **Intermediate:** Swap `Runner.run` for `Runner.run_sync` and remove the `async`/`await`/`asyncio.run`
   scaffolding — confirm the output is identical.
3. **Advanced:** Give `get_software_install_guide` a second parameter (e.g. `operating_system: str`) and
   update its docstring/type hints so the model asks a clarifying question when it's missing — observe how
   the Agents SDK's automatic schema generation reacts to the new parameter.
